DESCARGA TC40

In [ ]:
input("esperar:")

TRASLADO DE ARCHIVOS A RUTAS

In [ ]:
import win32com.client
import os
import shutil
import re
import pandas as pd
from sqlalchemy import create_engine
from datetime import datetime, timedelta

# Obtener el día de la semana actual (lunes = 0, martes = 1, ..., domingo = 6)
dia_actual = datetime.now().weekday()

# Conexión con Outlook
outlook = win32com.client.Dispatch("Outlook.Application").GetNamespace("MAPI")

# Seleccionar la bandeja de entrada
inbox = outlook.GetDefaultFolder(6)  # 6 representa la bandeja de entrada

# Buscar los correos
messages = inbox.Items
messages.Sort("[ReceivedTime]", True)  # Ordenar por fecha de recepción, descendente

# Expresión regular para el asunto (sin la parte de la fecha que puede variar)
asunto_regex = r"SAFE"

# Ruta donde quieres guardar el archivo adjunto
ruta_guardado = r"\\mc0816\05 Gestion de Informacion\7. Capacitaciones\Carga_Fruades\safe"

# Inicializar dataframe vacío para el archivo del correo
df_excel_correo = None

# Buscar el correo que coincide con el asunto
for message in messages:
    # Verificar si el mensaje es un correo electrónico
    if message.Class == 43:  # 43 representa correos electrónicos
        try:
            # Verificar si el asunto contiene el texto deseado
            if re.search(asunto_regex, message.Subject):
                print(f"Correo encontrado: {message.Subject}")
                
                # Descargar el archivo adjunto
                for attachment in message.Attachments:
                    if attachment.FileName.endswith('.xlsx'):
                        # Buscar fechas en el nombre del archivo (rango de fechas)
                        match = re.search(r"Safe (\d{2})\.(\d{2}) al (\d{2})\.(\d{2})", attachment.FileName, re.IGNORECASE)
                        if match:
                            dia_ini, mes_ini, dia_fin, mes_fin = match.groups()
                            anio_actual = datetime.now().year
                            nuevo_nombre = f"SAFE {dia_ini}{mes_ini}{anio_actual}al{dia_fin}{mes_fin}{anio_actual}.xlsx"
                        else:
                            # Buscar una única fecha en el nombre del archivo
                            match = re.search(r"Safe (\d{2})\.(\d{2})", attachment.FileName, re.IGNORECASE)
                            if match:
                                dia_ini, mes_ini = match.groups()
                                anio_actual = datetime.now().year
                                nuevo_nombre = f"SAFE {dia_ini}{mes_ini}{anio_actual}.xlsx"
                            else:
                                print(f"No se encontraron fechas en el nombre del archivo: {attachment.FileName}")
                                continue
                        
                        # Guardar el archivo con el nuevo nombre
                        archivo_safe = os.path.join(ruta_guardado, nuevo_nombre)
                        attachment.SaveAsFile(archivo_safe)
                        print(f"Archivo guardado como: {nuevo_nombre}")
                
                break  # Salir después de encontrar el correo y procesar el archivo
        except AttributeError as e:
            print(f"No se pudo acceder a la propiedad: {e}")


# Ruta del archivo original
ruta_original = r"C:\Users\mc2058\Downloads\TC40 Details.xlsx"

# Verificar si el archivo original existe antes de continuar
if os.path.exists(ruta_original):
# Lógica para renombrar dependiendo del día
    if dia_actual == 1:  # Si es lunesmc1937
        # Obtener las fechas de ayer y anteanteayer
        fecha_ayer = (datetime.now() - timedelta(days=1)).strftime("%d%m%Y")
        fecha_anteanteayer = (datetime.now() - timedelta(days=4)).strftime("%d%m%Y")
        
        # Extraer el nombre del archivo y su extensión
        nombre_archivo, extension = os.path.splitext(ruta_original)
        
        # Nuevo nombre del archivo para lunes
        nuevo_nombre = f"{os.path.basename(nombre_archivo)}_{fecha_anteanteayer} al {fecha_ayer}{extension}"

    else:  # Si es martes, miércoles, jueves o viernes
        # Obtener la fecha de ayer
        fecha_ayer = (datetime.now() - timedelta(days=1)).strftime("%d%m%Y")
        
        # Extraer el nombre del archivo y su extensión
        nombre_archivo, extension = os.path.splitext(ruta_original)
        
        # Nuevo nombre del archivo para martes a viernes
        nuevo_nombre = f"{os.path.basename(nombre_archivo)}_{fecha_ayer}{extension}"

    # Ruta donde deseas mover el archivo
    ruta_destino = r"\\mc0816\05 Gestion de Informacion\8. TC40"

    # Asegurarte de que la ruta de destino existe
    os.makedirs(ruta_destino, exist_ok=True)

    # Componer la ruta completa del nuevo archivo
    archivo_tc40 = os.path.join(ruta_destino, nuevo_nombre)

    # Mover y renombrar el archivo usando shutil
    shutil.move(ruta_original, archivo_tc40)
    print(f"Archivo renombrado y movido a: {archivo_tc40}")
    
else:
    print(f"El archivo original no existe: {ruta_original}")


GENERACION DE FRAUDES

In [ ]:
import os
from pathlib import Path
import pandas as pd
from datetime import datetime
import numpy as np
import unidecode

# Función para convertir los números de serie a fecha, dejando el resto intacto
def convertir_fecha_txn(valor):
    try:
        valor = str(valor).strip()  # Elimina espacios
        numero = pd.to_numeric(valor, errors='coerce')
        if not np.isnan(numero):
            return pd.to_datetime(numero - 25569, unit='D', origin='unix').strftime('%d/%m/%Y')
        else:
            # Intentar convertir directamente con formato texto
            return pd.to_datetime(valor, format='%d-%b-%Y', errors='coerce').strftime('%d/%m/%Y')
    except:
        return ''

# archivo_safe =r"\\mc0816\05 Gestion de Informacion\7. Capacitaciones\Carga_Fruades\safe\SAFE 13102025al14102025.xlsx"
# archivo_tc40 =r"\\mc0816\05 Gestion de Informacion\8. TC40\TC40 Details_01082025 al 12082025.xlsx"

# Función para obtener la fecha de creación de un archivo
def obtener_fecha_archivo(ruta):
    return datetime.fromtimestamp(os.path.getctime(ruta))

# Obtener las fechas de creación de los archivos
fecha_tc40 = obtener_fecha_archivo(archivo_tc40).date()
fecha_safe = obtener_fecha_archivo(archivo_safe).date()

# Comparar las fechas de los archivos
if fecha_tc40 == fecha_safe:
    # Si las fechas coinciden, procesar ambos archivos
    print(f"Ambos archivos tienen la misma fecha: {fecha_tc40}. Procesando ambos archivos...")

    # Procesar archivo TC40
    df_tc40 = pd.read_excel(archivo_tc40, skiprows=5, dtype=str)
    df_tc40['Acquirer Reference Number'] = df_tc40['Acquirer Reference Number'].replace(np.nan, '', regex=True)
    df_tc40['Card Account Number Masked'] = df_tc40['Card Account Number Masked'].str.replace(' ', '')
    df_tc40['Card Account Number Masked'] = df_tc40['Card Account Number Masked'].replace(np.nan, '', regex=True)
    df_tc40['Fraud Main Date'] = df_tc40['Fraud Main Date'].apply(convertir_fecha_txn)
    df_tc40['Card Account Number Masked'] = df_tc40['Card Account Number Masked'].str.replace('x', '*')
    df_tc40['llave'] = df_tc40['Card Account Number Masked'].astype(str) + ";" + df_tc40['Acquirer Reference Number'].astype(str)
    df_tc40['Fuente'] = 'F:TC40'
    df_tc40['Fraud Amount (Local Currency)'] = df_tc40['Fraud Amount (Local Currency)'].astype(float).map('{:.2f}'.format)
    df_tc40['Fraud Amount'] = df_tc40['Fraud Amount'].astype(float).map('{:.2f}'.format)
    df_tc40 = df_tc40.drop(columns=['Card Account Number Masked', 'Acquirer Reference Number'])
    df_tc40 = df_tc40.iloc[:-2, :]
    # df_tc40['Fraud Main Date'] = pd.to_datetime(df_tc40['Fraud Main Date'], errors='coerce').dt.strftime('%d/%m/%Y')
    df_tc40_seleccionado = df_tc40[['llave', 'Fraud Amount', 'Fraud Amount (Local Currency)', 'Fraud Type', 'Fuente', 'Fraud Main Date']]

    # Procesar archivo SAFE
    df_safe = pd.read_excel(archivo_safe, dtype=str)
    # df_safe = pd.read_excel(archivo_safe, skiprows=3, dtype=str)
    # df_safe = pd.read_excel(archivo_safe, sheet_name='Details', dtype=str)
    df_safe['Acquirer Reference Number'] = df_safe['Acquirer Reference Number'].replace(np.nan, '', regex=True)
    df_safe['Card Number'] = df_safe['Card Number'].replace(np.nan, '', regex=True)
    df_safe['llave'] = df_safe['Card Number'].astype(str) + ";" + df_safe['Acquirer Reference Number'].astype(str)
    df_safe['Fuente'] = 'F:Safe'
    df_safe['Personalizado'] = df_safe['Fraud Type'].str[:2]
    df_safe['Personalizado'] = pd.to_numeric(df_safe['Personalizado'], errors='coerce', downcast='integer')
    # df_safe['Date (Entered Date)'] = df_safe['Date (Entered Date)'].apply(convertir_fecha_txn)
    # df_safe['US Trans Amount'] = df_safe['US Trans Amount'].astype(float).map('{:.2f}'.format)
    df_safe['US Trans Amount'] = df_safe['US Trans Amount'].str.replace('$', '', regex=False).astype(float).map('{:.2f}'.format)
    df_safe['Transaction Amount'] = df_safe['Transaction Amount'].astype(float).map('{:.2f}'.format)
    df_safe = df_safe.drop(columns=["Fraud Type", "Card Number", "Acquirer Reference Number", "US Bill Amount", "Bill Amount"])
    # df_safe['Date (Entered Date)'] = pd.to_datetime(df_safe['Date (Entered Date)'], errors='coerce').dt.strftime('%d/%m/%Y')
    df_safe['Entered Date'] = pd.to_datetime(df_safe['Entered Date'], errors='coerce').dt.strftime('%d/%m/%Y')
    # df_safe_seleccionado = df_safe[['llave', 'US Trans Amount', 'Transaction Amount', 'Personalizado', 'Fuente', 'Date (Entered Date)']]
    df_safe_seleccionado = df_safe[['llave', 'US Trans Amount', 'Transaction Amount', 'Personalizado', 'Fuente', 'Entered Date']]
    df_safe_seleccionado = df_safe_seleccionado.rename(columns={
        'US Trans Amount': 'Fraud Amount',
        # 'Transaction Amount': 'Fraud Amount (Local Currency)',
        'Transaction Amount': 'Fraud Amount (Local Currency)',
        'Personalizado': 'Fraud Type',
        'Entered Date': 'Fraud Main Date'
        # 'Date (Entered Date)': 'Fraud Main Date'
    })

    # Concatenar ambos dataframes
    df_concatenado = pd.concat([df_tc40_seleccionado, df_safe_seleccionado], axis=0, ignore_index=True)

elif fecha_tc40 > fecha_safe:
    # Si el archivo TC40 tiene la fecha más reciente
    print(f"El archivo TC40 tiene la fecha más reciente: {fecha_tc40}. Procesando solo TC40...")

    # Procesar archivo TC40
    df_tc40 = pd.read_excel(archivo_tc40, skiprows=5, dtype=str)
    df_tc40['Acquirer Reference Number'] = df_tc40['Acquirer Reference Number'].replace(np.nan, '', regex=True)
    df_tc40['Card Account Number Masked'] = df_tc40['Card Account Number Masked'].str.replace(' ', '')
    df_tc40['Card Account Number Masked'] = df_tc40['Card Account Number Masked'].replace(np.nan, '', regex=True)
    df_tc40['Fraud Main Date'] = df_tc40['Fraud Main Date'].apply(convertir_fecha_txn)
    df_tc40['Card Account Number Masked'] = df_tc40['Card Account Number Masked'].str.replace('x', '*')
    df_tc40['llave'] = df_tc40['Card Account Number Masked'].astype(str) + ";" + df_tc40['Acquirer Reference Number'].astype(str)
    df_tc40['Fuente'] = 'F:TC40'
    df_tc40['Fraud Amount (Local Currency)'] = df_tc40['Fraud Amount (Local Currency)'].astype(float).map('{:.2f}'.format)
    df_tc40['Fraud Amount'] = df_tc40['Fraud Amount'].astype(float).map('{:.2f}'.format)
    df_tc40 = df_tc40.drop(columns=['Card Account Number Masked', 'Acquirer Reference Number'])
    df_tc40 = df_tc40.iloc[:-2, :]
    # df_tc40['Fraud Main Date'] = pd.to_datetime(df_tc40['Fraud Main Date'], errors='coerce').dt.strftime('%d/%m/%Y')
    df_concatenado = df_tc40[['llave', 'Fraud Amount', 'Fraud Amount (Local Currency)', 'Fraud Type', 'Fuente', 'Fraud Main Date']]

else:
    # Si el archivo SAFE tiene la fecha más reciente
    print(f"El archivo SAFE tiene la fecha más reciente: {fecha_safe}. Procesando solo SAFE...")

    # Procesar archivo SAFE
    # df_safe = pd.read_excel(archivo_safe, skiprows=3, dtype=str)
        # Procesar archivo SAFE
    df_safe = pd.read_excel(archivo_safe, dtype=str)
    # df_safe = pd.read_excel(archivo_safe, skiprows=3, dtype=str)
    # df_safe = pd.read_excel(archivo_safe, sheet_name='Details', dtype=str)
    df_safe['Acquirer Reference Number'] = df_safe['Acquirer Reference Number'].replace(np.nan, '', regex=True)
    # Eliminar columnas si existen
    cols_a_eliminar = ['Auth Code', 'DI Score', 'DI Score Reason']
    df_safe = df_safe.drop(columns=[c for c in cols_a_eliminar if c in df_safe.columns], errors='ignore')
    df_safe['Acquirer Reference Number'] = df_safe['Acquirer Reference Number'].replace(np.nan, '', regex=True)
    df_safe['Card Number'] = df_safe['Card Number'].replace(np.nan, '', regex=True)
    df_safe['llave'] = df_safe['Card Number'].astype(str) + ";" + df_safe['Acquirer Reference Number'].astype(str)
    df_safe['Fuente'] = 'F:Safe'
    df_safe['Personalizado'] = df_safe['Fraud Type'].str[:2]
    df_safe['Personalizado'] = pd.to_numeric(df_safe['Personalizado'], errors='coerce', downcast='integer')
    # df_safe['Date (Entered Date)'] = df_safe['Date (Entered Date)'].apply(convertir_fecha_txn)
    df_safe['US Trans Amount'] = df_safe['US Trans Amount'].astype(float).map('{:.2f}'.format)
    df_safe['Transaction Amount'] = df_safe['Transaction Amount'].astype(float).map('{:.2f}'.format)
    df_safe = df_safe.drop(columns=["Fraud Type", "Card Number", "Acquirer Reference Number", "US Bill Amount", "Bill Amount"])
    # df_safe['Date (Entered Date)'] = pd.to_datetime(df_safe['Date (Entered Date)'], errors='coerce').dt.strftime('%d/%m/%Y')
    df_safe['Entered Date'] = pd.to_datetime(df_safe['Entered Date'], errors='coerce').dt.strftime('%d/%m/%Y')
    # df_concatenado = df_safe[['llave', 'US Trans Amount', 'Transaction Amount', 'Personalizado', 'Fuente', 'Date (Entered Date)']]
    df_concatenado = df_safe[['llave', 'US Trans Amount', 'Transaction Amount', 'Personalizado', 'Fuente', 'Entered Date']]
    df_concatenado = df_concatenado.rename(columns={
        'US Trans Amount': 'Fraud Amount',
        # 'Transaction Amount': 'Fraud Amount (Local Currency)',
        'Transaction Amount': 'Fraud Amount (Local Currency)',
        'Personalizado': 'Fraud Type',
        'Entered Date': 'Fraud Main Date'
        # 'Date (Entered Date)': 'Fraud Main Date'
    })

# Guardar el dataframe final en un archivo Excel
ruta_salida = r'\\mc0816\05 Gestion de Informacion\7. Capacitaciones\JhonVera\PR0048\Documentos\df_concatenado.xlsx'
df_concatenado.to_excel(ruta_salida, index=False)

print(f"El dataframe procesado ha sido guardado en {ruta_salida}")


GENERACION DE ARD SAFE

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

# Obtener la columna 'llave'
col_llave = df_concatenado['llave']

# Crear una lista donde el primer valor es el primer valor de 'llave'
valores = [col_llave.iloc[0]] + col_llave.tolist()

# Convertir a un dataframe sin encabezado
df_llave = pd.DataFrame(valores)

# Obtener la fecha de ayer en formato 'yyyymmdd'
fecha_ayer = (datetime.now() - timedelta(days=1)).strftime('%Y%m%d')

# Ruta de guardado
nombre_archivo = f"ARD_SAFE_{fecha_ayer}.csv"
ruta_destino = r'\\mc0016\FTPTRANSFER\Interno\RIESGOS\ARD\in' + "\\" + nombre_archivo

# Guardar el archivo como CSV sin encabezado y sin índice
df_llave.to_csv(ruta_destino, header=False, index=False)

print(f"Archivo guardado en: {ruta_destino}")

PROCESAR CON EL AS400 ..............

In [ ]:
input("esperar PROCESAR CON EL AS400_:")

In [ ]:
import os
from pathlib import Path
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import unidecode

# Función para convertir los números de serie a fecha, dejando el resto intacto
def convertir_fecha_txn(valor):
    try:
        numero = pd.to_numeric(valor, errors='coerce')
        if not np.isnan(numero):
            return pd.to_datetime(numero - 25569, unit='D', origin='unix').strftime('%d/%m/%Y')
        else:
            return valor
    except:
        return valor

# Obtener la fecha de ayer
fecha_ayer = datetime.now() - timedelta(days=1)

# Formatear la fecha en el formato deseado
fecha_ayer_str = fecha_ayer.strftime('%Y%m%d')  # Formato: YYYYMMDD
anio = fecha_ayer.strftime('%Y')  # Año: YYYY
mes = fecha_ayer.strftime('%m')   # Mes: MM
dia = fecha_ayer.strftime('%d')   # Día: DD

# Construir la ruta del archivo
ruta_archivo = f"\\\\MC0014\\Reporte\\{anio}\\{mes}\\{dia}\\MCPERU_PCI\\ARD_SAFE_PROCESSED_{fecha_ayer_str}.csv"
# ruta_archivo = r'\\MC0014\Reporte\2024\11\28\MCPERU_PCI\ARD_SAFE_PROCESSED_20241128.csv'

# Dataframe concanteado
ruta_salida = r'\\mc0816\05 Gestion de Informacion\7. Capacitaciones\JhonVera\PR0048\Documentos\df_concatenado.xlsx'

# Verificar si el archivo existe
if os.path.exists(ruta_archivo):
    print(f"El archivo existe: {ruta_archivo}")
else:
    print(f"El archivo no se encontró: {ruta_archivo}")

# Definir los nombres de las columnas
nombres_columnas = [f'column{i+1}' for i in range(21)]

# Leer el archivo CSV sin encabezados y agregar los nombres de las columnas
df_ARD = pd.read_csv(ruta_archivo, sep=';', dtype=str, encoding='ISO-8859-1', names=nombres_columnas)
df_fraudes = pd.read_excel(ruta_salida,dtype=str)

df_fraudes['Fraud Amount'] = df_fraudes['Fraud Amount'].astype(str).str.replace(',', '', regex=False)
df_fraudes['Fraud Amount (Local Currency)'] = df_fraudes['Fraud Amount (Local Currency)'].astype(str).str.replace(',', '', regex=False)

# Función para formatear los números
def format_fraud_amount(value):
    # Si termina en .00
    if value.endswith('.00'):
        return value[:-3]  # Elimina los últimos 3 caracteres (.00)
    # Si termina en .0
    elif value.endswith('.0'):
        return value[:-2]  # Elimina los últimos 2 caracteres (.0)
    # Si termina en .x0 (x es cualquier dígito)
    elif value.endswith('0') and '.' in value:
        return value[:-1]  # Elimina el último 0
    # Si es un número con más de un decimal, mantenerlo como está
    elif '.' in value and value.count('.') == 1:
        return value  # Mantiene números como 92.82 sin cambios
    else:
        return value  # Retorna el valor sin cambios si no cumple con las condiciones

# Aplicar la función a ambas columnas
df_fraudes['Fraud Amount'] = df_fraudes['Fraud Amount'].apply(format_fraud_amount)
df_fraudes['Fraud Amount (Local Currency)'] = df_fraudes['Fraud Amount (Local Currency)'].apply(format_fraud_amount)


df_ARD['column11'] = df_ARD['column11'].replace("'", "", regex=True)
df_ARD['column1'] = df_ARD['column1'].replace("'", "", regex=True)

df_ARD['NewColumn'] = (
    df_ARD['column1'].str[:6] +  # Tomar los primeros 6 caracteres
    "******" +                    # Agregar asteriscos
    df_ARD['column1'].str[-4:] + # Tomar los últimos 4 caracteres
    ";" +                          # Agregar punto y coma
    df_ARD['column11']           # Agregar Column11
)

# Crear y limpiar las columnas solicitadas
df_ARD['Tarjeta_Encriptada'] = df_ARD['column1'].str.strip()  # LIMPIAR(Column1)
df_ARD['Autorizacion'] = df_ARD['column2'].str.strip()         # LIMPIAR(Column2)
df_ARD['Establecimiento'] = df_ARD['column3'].str.strip()      # LIMPIAR(Column3)
df_ARD['VOU'] = df_ARD['column4'].str.strip()                  # LIMPIAR(Column4)
df_ARD['Fecha transacción'] = df_ARD['column5'].str.strip()    # LIMPIAR(Column5)
df_ARD['Moneda'] = df_ARD['column6'].str.strip()               # LIMPIAR(Column6)
df_ARD['Transaccion'] = df_ARD['column7'].str.strip()          # LIMPIAR(Column7)
df_ARD['Nombre comer'] = df_ARD['column8'].str.strip()         # LIMPIAR(Column8)
df_ARD['Fecha proceso'] = df_ARD['column9'].str.strip()        # LIMPIAR(Column9)
df_ARD['PDARER'] = df_ARD['column10'].str.strip()              # LIMPIAR(Column10)
df_ARD['PAREN (ARD/ARN)'] = df_ARD['column11'].str.strip()     # LIMPIAR(Column11)
df_ARD['Nro. Documento'] = df_ARD['column12'].str.strip()      # LIMPIAR(Column12)
df_ARD['ECOM'] = df_ARD['column13'].str.strip()                # LIMPIAR(Column13)
df_ARD['Terminal'] = df_ARD['column14'].str.strip()            # LIMPIAR(Column14)
df_ARD['Entry mode'] = df_ARD['column15'].str.strip()          # LIMPIAR(Column15)
df_ARD['Referencia'] = df_ARD['column16'].str.strip()          # LIMPIAR(Column16)
df_ARD['Importe'] = df_ARD['column17'].str.strip()             # LIMPIAR(Column17)
df_ARD['ID_Transaction'] = df_ARD['column18'].str.strip()      # LIMPIAR(Column18)
df_ARD['Aprobado por'] = df_ARD['column19'].str.strip()        # LIMPIAR(Column19)
df_ARD['Fecha'] = df_ARD['column20'].str.strip()               # LIMPIAR(Column20)
df_ARD['Hora'] = df_ARD['column21'].str.strip()
df_ARD['Bin'] = df_ARD['column1'].str[:6]   # Tomar los primeros 6 caracteres
df_ARD['Sufix'] = df_ARD['column1'].str[-4:]  # Tomar los últimos 4 caracteres

# Realizamos el merge de df_ARD con df_fraudes basado en la columna 'NewColumn' y 'llave'
df_ARD = df_ARD.merge(df_fraudes[['llave', 'Fraud Amount', 'Fraud Amount (Local Currency)', 'Fraud Type', 'Fuente', 'Fraud Main Date']],
                      left_on='NewColumn', right_on='llave', how='left')

# Asignamos las nuevas columnas con los nombres correctos
df_ARD['Fecha_Carga'] = df_ARD['Fraud Main Date'].astype(str)
df_ARD['Monto'] = df_ARD['Fraud Amount'].astype(str)
df_ARD['Importe_Soles'] = df_ARD['Fraud Amount (Local Currency)'].astype(str)
df_ARD['Tipo_Fraude'] = df_ARD['Fraud Type'].astype(str)
df_ARD['Fuente'] = df_ARD['Fuente'].astype(str)

# Eliminamos la columna 'llave' y otras columnas no necesarias si es necesario
df_ARD.drop(['llave', 'Fraud Main Date', 'Fraud Amount', 'Fraud Amount (Local Currency)', 'Fraud Type'], axis=1, inplace=True)



GENERAR TXT DE FRAUDES

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import shutil

# Obtener la fecha de ayer en el formato ddmmyyyy
fecha_ayer = (datetime.now() - timedelta(days=1)).strftime('%Y%m%d')

# Ruta y nombre del archivo
ruta_archivo = f"\\\\mc0816\\05 Gestion de Informacion\\1. Carga de Informacion\\1. Acumulables\\FRAUDES\\TC40-Safe_{fecha_ayer}.txt"

# Seleccionamos solo las columnas necesarias
columnas_seleccionadas = [
    'Tarjeta_Encriptada', 'Autorizacion', 'Establecimiento', 'VOU', 'Fecha transacción', 'Moneda',
    'Transaccion', 'Nombre comer', 'Fecha proceso', 'PDARER', 'PAREN (ARD/ARN)', 'Nro. Documento',
    'ECOM', 'Terminal', 'Entry mode', 'Referencia', 'Importe', 'ID_Transaction', 'Aprobado por',
    'Fecha', 'Hora', 'Bin', 'Sufix', 'Fecha_Carga', 'Monto', 'Importe_Soles', 'Tipo_Fraude', 'Fuente'
]

# Filtrar las columnas del dataframe
df_filtrado = df_ARD[columnas_seleccionadas]

# Guardar en archivo .txt usando tabulaciones como separador
df_filtrado.to_csv(ruta_archivo, sep='\t', index=False, header=True)

print(f"Archivo .txt guardado en: {ruta_archivo}")

# # Definir rutas
# ruta_1 = ruta_archivo  # Archivo original generado
# ruta_2= r'D:\JhonVera\CargaAutomatica\Fraudes TC40 Safe\TC40-Safe.txt'  # Nueva ubicación con nuevo nombre

# # Copiar y renombrar el archivo
# shutil.copy(ruta_1, ruta_2)

# print(f"Archivo copiado a: {ruta_destino}")
